In [ ]:
from copy import deepcopy
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from Bio import Phylo

FIG = Path("figures")
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.family": "DejaVu Sans"})


def assembly_of(name):
    return str(name).split("|", 1)[0]


def support_of(clade):
    text = str(clade.name) if clade.name else ""
    if "/" not in text:
        return None
    left, right = text.split("/", 1)
    try:
        return float(left), float(right)
    except ValueError:
        return None


meta = pd.read_csv("results/sequences/colap_clean_metadata.csv", dtype=str).set_index("assembly_accession")
tree = Phylo.read("results/phylogeny/colap_ml.treefile", "newick")
display = deepcopy(tree)
for tip in display.get_terminals():
    tip.name = assembly_of(tip.name)
display.root_at_midpoint()
display.ladderize()
protected = {
    "GCF_000012325.1", "GCF_002836245.1", "GCF_000764225.1", "GCF_007954275.1",
    "GCF_007097525.1", "GCA_002162925.1", "GCF_000764205.1", "GCA_000712155.1", "GCA_002734005.2",
}
chosen = []
for clade in display.get_nonterminals():
    if clade is display.root:
        continue
    pair = support_of(clade)
    names = [tip.name for tip in clade.get_terminals()]
    if pair is None or pair[0] < 80 or pair[1] < 95 or len(names) < 8 or protected.intersection(names):
        continue
    chosen.append(clade)
chosen.sort(key=lambda clade: len(clade.get_terminals()), reverse=True)
covered = set()
collapsed = []
for clade in chosen:
    names = {tip.name for tip in clade.get_terminals()}
    if names & covered:
        continue
    collapsed.append(clade)
    covered |= names
for clade in collapsed:
    names = [tip.name for tip in clade.get_terminals()]
    genera = meta.loc[names, "genus"].value_counts()
    label = ", ".join(f"{genus} {count}" for genus, count in genera.items())
    pair = support_of(clade)
    clade.clades = []
    clade.name = f"{label} (n={len(names)}; {pair[0]:.1f}/{pair[1]:.0f})"


def place(clade, cursor, x):
    clade.x = x
    if not clade.clades:
        clade.y = cursor[0]
        cursor[0] += 1
        return
    for child in clade.clades:
        place(child, cursor, x + float(child.branch_length or 0))
    clade.y = sum(child.y for child in clade.clades) / len(clade.clades)


place(display.root, [0], 0.0)
figure, axis = plt.subplots(figsize=(11.2, 10.4))


def draw(clade):
    for child in clade.clades:
        axis.plot([clade.x, clade.x], [clade.y, child.y], color="#444444", lw=0.7)
        axis.plot([clade.x, child.x], [child.y, child.y], color="#444444", lw=0.7)
        pair = support_of(child)
        if pair is not None and pair[0] >= 80 and pair[1] >= 95:
            axis.scatter([child.x], [child.y], s=16, c="black", zorder=3)
        draw(child)


draw(display.root)
for tip in display.get_terminals():
    axis.text(tip.x + 0.01, tip.y, tip.name, va="center", ha="left", fontsize=7)
axis.set_axis_off()
axis.set_title("ColAP maximum-likelihood tree")
figure.tight_layout()
figure.savefig(FIG / "colap_phylogeny.png", dpi=300)
plt.close()

comparison = pd.read_csv("results/comparison/tree_comparison_statistics.csv").iloc[0]
figure, axis = plt.subplots(figsize=(6.2, 4.2))
labels = ["Concordant", "Conflicting"]
values = [comparison.supported_concordant, comparison.supported_conflicting]
axis.bar(labels, values, color=["#0072B2", "#D55E00"])
axis.set_ylabel("Well-supported ColAP splits")
axis.set_title(f"Gene tree vs species tree, {int(comparison.matched)} shared strains")
for index, value in enumerate(values):
    axis.text(index, value + 0.3, str(int(value)), ha="center")
figure.tight_layout()
figure.savefig(FIG / "gene_vs_species.png", dpi=300)
plt.close()

temperature = pd.read_csv("results/temperature/pgls_h1.csv")
labels = {"proline_fraction": "Proline fraction", "gravy": "GRAVY", "arg_over_arg_plus_lys": "Arg/(Arg+Lys)"}
figure, axis = plt.subplots(figsize=(7.2, 4.4))
ypos = np.arange(len(temperature))[::-1]
axis.axvline(0, color="#888888", lw=0.8)
axis.errorbar(
    temperature.beta_per_sd,
    ypos,
    xerr=[temperature.beta_per_sd - temperature.ci95_low_per_sd, temperature.ci95_high_per_sd - temperature.beta_per_sd],
    fmt="o",
    color="#222222",
    capsize=3,
)
axis.set_yticks(ypos)
axis.set_yticklabels([f"{labels[row.trait]}\nHolm p = {row.p_holm:.3f}" for row in temperature.itertuples(index=False)])
axis.set_xlabel("PGLS coefficient, degrees C per 1 SD")
axis.set_title(f"ColAP composition and OGT, n = {int(temperature.n.iloc[0])}")
figure.tight_layout()
figure.savefig(FIG / "h1_temperature.png", dpi=300)
plt.close()

fit = pd.read_csv("results/hypothesis/h2_distance.csv").iloc[0]
points = pd.read_csv("results/hypothesis/h2_partial_residuals.csv")
domain_color = {
    "2.60.40.1730": "#0072B2",
    "3.30.2010.30": "#E69F00",
    "1.10.390.10": "#009E73",
    "1.25.40.320": "#CC79A7",
    "none": "#7F7F7F",
}
domain_label = {
    "2.60.40.1730": "Ig-like",
    "3.30.2010.30": "zincin-like",
    "1.10.390.10": "protease domain 2",
    "1.25.40.320": "C-terminal M1",
    "none": "outside CATH",
}
figure, axis = plt.subplots(figsize=(6.67, 7.5))
for ident, frame in points.groupby("cath_id", sort=False):
    axis.scatter(frame.distance_A, frame.partial_residual, s=36, alpha=0.55, c=domain_color[ident], label=domain_label[ident])
grid = np.linspace(points.distance_A.min(), points.distance_A.max(), 200)
z_grid = (grid - fit.distance_mean_A) / fit.distance_sd_A
axis.plot(grid, fit.beta_distance * z_grid, color="#222222", lw=2.6)
axis.fill_between(grid, fit.block_bootstrap_ci95_low * z_grid, fit.block_bootstrap_ci95_high * z_grid, color="#222222", alpha=0.15)
axis.set_xlabel("Distance to functional center (A)")
axis.set_ylabel("Adjusted evolutionary rate (partial residual)")
axis.set_title("Evolutionary rates increase with distance\nfrom the functional center")
axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False, fontsize=9)
figure.subplots_adjust(left=0.16, right=0.97, bottom=0.24, top=0.90)
figure.savefig(FIG / "h2_rate_vs_distance.png", dpi=300)
plt.close()

effects = pd.read_csv("data/robustness/h2_effects.csv")
keep = ["baseline", "substitution_model", "tree", "alternative_method"]
effects = effects[effects.family.isin(keep)].copy()
effects["family"] = pd.Categorical(effects.family, keep, ordered=True)
effects = effects.sort_values("family")
names = {
    "baseline": "LEISR WAG, topology 3A",
    "substitution_model": "LEISR LG, topology 3A",
    "tree": "LEISR WAG, control topology 3B",
    "alternative_method": "Rate4Site WAG, topology 3A",
}
figure, axis = plt.subplots(figsize=(8.4, 4.6))
ypos = np.arange(len(effects))[::-1]
axis.axvline(0, color="#888888", lw=0.8)
axis.errorbar(
    effects.estimate,
    ypos,
    xerr=[effects.estimate - effects.ci95_low, effects.ci95_high - effects.estimate],
    fmt="o",
    color="#222222",
    capsize=3,
)
axis.set_yticks(ypos)
axis.set_yticklabels([names[family] for family in effects.family])
axis.set_xlabel("Distance coefficient on log(1 + rate)")
axis.set_title("Distance coefficient under alternative rate estimates")
figure.tight_layout()
figure.savefig(FIG / "h2_robustness.png", dpi=300)
plt.close()
